In [ ]:
%display latex

In [ ]:
SIZE = 4 # 8 # 16 # 32
K = QQ

Kx = PolynomialRing(K, x, SIZE) # K[*SR.var('X', 4)] # K[*(f"y{i}" for i in range(5))]
Kxy.<A> = Kx[]

display(
    Kx,
    Kxy,
    xs_poly := A**SIZE + Kxy(xs := Kx.gens()),
)

In [ ]:
C_xs = (
    zero_vector(Kx, SIZE - 1)
    .column()
    .augment(identity_matrix(Kx, SIZE - 1))
    .stack(-vector(xs))
    .T
)  # block_matrix
display(C_xs, C_poly := C_xs.charpoly(var=A))
assert C_xs == companion_matrix(xs_poly)  # Matrix.companion()
assert (
    xs_poly == C_poly == C_xs.T.charpoly(var=A)
), "Companion-matrix construction failed to yield proper properties!"
assert xs_poly.subs(C_xs) == 0, "Cayley–Hamilton theorem did not hold!"

In [ ]:
from tqdm.auto import tqdm

print("Powers of Companion Matrix C_xs")
display(
    table(
        (
            C_pows := [
                [i, xi, C_i, C_xi := xi * C_i]
                for i, (xi, C_i) in enumerate(
                    zip(xs, tqdm(C_xs.powers(SIZE)), strict=True)
                )
            ]
        )[: (None if get_verbose() > 0 else 1)]
    ).transpose()
)
assert (C_lc := C_xs**SIZE) == -(
    C_tail := sum(C_pow[3] for C_pow in C_pows)
), f"{C_lc=}, {C_tail=}"
if get_verbose() > 0:
    display(C_lc, C_tail)

In [ ]:
from sage.misc.verbose import verbose
from tqdm.auto import tqdm, trange

print("Natural basis should cycle under shifted power series; hashes should match!")

results = {}
for e_i, e_v in enumerate(tqdm(identity_matrix(K, SIZE).columns(copy=False))):
    verbose(f"{e_i=}, {e_v=}", 1)
    for j in trange(-SIZE, 2 * SIZE, leave=False):
        (e_out := (C_xs ** (k := j - e_i)) * e_v).set_immutable()
        verbose(f"{j=}, {k=}, {(e_out_hash := hash(e_out))}", 2)
        assert results.setdefault(j, e_out_hash) == e_out_hash, "Mismatch found!"
    verbose("---", 2)

print("Held!")